
# 応用問題: 勾配降下法でロジスティック回帰を学習する

## 目標

データから重みを**学習**する最も基本的な機械学習である **ロジスティック回帰** を, **勾配降下法 (gradient descent)** で訓練する。学習が進むにつれて**正解率が上がっていく**様子を観察する。

並列化対象は AI 学習の心臓部, **全サンプルにわたる予測と勾配の和** (行列積 + reduction) である。

## 課題

合成データを使う:

- 真の重み `w_true[D]` を乱数で決め, 各サンプル `i` の特徴 `x_i[d] = draw_rand01(i,d) - 0.5` を生成。
- ラベル `y_i = (w_true・x_i > 0) ? 1 : 0` (**線形分離可能**)。

ロジスティック回帰:

- 予測 `p = sigmoid(w・x)`, 損失は二値クロスエントロピー。
- 勾配 `grad[d] = (1/N) Σ_i (sigmoid(w・x_i) - y_i) * x_i[d]`。
- `w` を 0 から始め, 各エポックで `w[d] -= lr * grad[d]`。

各エポックで一番重いのは **全 `N` サンプルにわたる O(N·D) の計算**。これを 2 つのループに分けてある:

1. **サンプルのループ** (誤差 `err[i] = p - y_i` の計算 + 損失・正解数の集計)。各サンプルは独立。
   - C++: `#pragma omp parallel for reduction(+:loss,correct)`
   - Fortran: `!$omp parallel do private(...) reduction(+:loss,correct)` … `!$omp end parallel do`

これが `TODO` の並列化箇所である。

2. 勾配 `grad[d]` のループは特徴 `d` で並列化済み (`#pragma omp parallel for` / `!$omp parallel do`, 競合なし)。サンプルにわたる和を `err[]` に分けたので reduction の競合を避けている。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore regression.cpp -o regression.exe

# Fortran
nvfortran -fast -mp=multicore regression.f90 -o regression.exe
```

引数: サンプル数 `N` (既定 200000), 特徴次元 `D` (既定 20), エポック数 `E` (既定 200), 学習率 `lr` (既定 1.0)。

```
OMP_NUM_THREADS=4 ./regression.exe 200000 20 200 1.0
```

## 期待される結果

```
epoch   0: loss=0.6931, acc=...%
epoch  50: loss=..., acc=...%
...
最終: N=200000, D=20, epochs=200, loss=..., acc=99..%
elapsed = ... sec
```

- 学習が進むと **損失が下がり, 正解率が上がる** (線形分離可能なデータなので最終的に **95% を大きく超える**, ほぼ 99%+)。
- `OMP_NUM_THREADS` を増やすと `elapsed` が短くなる (台数効果)。結果 (正解率) は本質的に同じになる。
- (発展) 内側の `w・x` 積を SIMD 化, あるいは GPU にオフロードして更に高速化できる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ regression.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (合成データ生成用): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

static inline double sigmoid(double z) { return 1.0 / (1.0 + exp(-z)); }

/* 勾配降下法でロジスティック回帰を学習する。
   予測 p = sigmoid(w・x)。損失は二値クロスエントロピー。
   勾配 grad[d] = (1/N) Σ_i (sigmoid(w・x_i) - y_i) * x_i[d]。
   w を 0 から始め, 各エポックで w[d] -= lr * grad[d] と更新する。
   合成データは線形分離可能なので, 学習が進むと正解率が上がっていくのを観察できる。
   並列化対象は「全サンプルにわたる予測・誤差の和」(行列積 + reduction)。 */
int main(int argc, char ** argv) {
  long N = (argc > 1 ? atol(argv[1]) : 200000);  /* サンプル数 */
  int  D = (argc > 2 ? atoi(argv[2]) : 20);      /* 特徴次元 */
  int  E = (argc > 3 ? atoi(argv[3]) : 200);     /* エポック数 */
  double lr = (argc > 4 ? atof(argv[4]) : 1.0);  /* 学習率 */

  /* 真の重み w_true (= 学習で復元したい正解), 範囲 [-1,1)。 */
  double * w_true = (double *)malloc(sizeof(double) * D);
  for (int d = 0; d < D; d++) w_true[d] = draw_rand01(d, 7) * 2.0 - 1.0;

  /* 特徴 x[i][d] (中心化), ラベル y[i] = (w_true・x_i > 0)。線形分離可能。 */
  double * X = (double *)malloc(sizeof(double) * N * D);
  int    * y = (int *)malloc(sizeof(int) * N);
  for (long i = 0; i < N; i++) {
    double score = 0.0;
    for (int d = 0; d < D; d++) {
      double xv = draw_rand01(i, d) - 0.5;
      X[i * D + d] = xv;
      score += w_true[d] * xv;
    }
    y[i] = (score > 0.0) ? 1 : 0;
  }

  double * w    = (double *)malloc(sizeof(double) * D);
  double * grad = (double *)malloc(sizeof(double) * D);
  double * err  = (double *)malloc(sizeof(double) * N);  /* 各サンプルの誤差 (p - y) */
  for (int d = 0; d < D; d++) w[d] = 0.0;

  double loss = 0.0;
  long correct = 0;
  double t0 = omp_get_wtime();
  for (int ep = 0; ep < E; ep++) {
    loss = 0.0;
    correct = 0;
    /* 各サンプルの予測 p = sigmoid(w・x_i), 誤差 err[i] = p - y_i,
       損失・正解数を集計する。各サンプルは独立なので並列化できる。 */
    // TODO: サンプルのループを #pragma omp parallel for reduction(+:loss,correct) で並列化せよ.
    for (long i = 0; i < N; i++) {
      double z = 0.0;
      for (int d = 0; d < D; d++) z += w[d] * X[i * D + d];
      double p = sigmoid(z);
      err[i] = p - (double)y[i];
      double eps = 1e-12;
      loss -= (y[i] ? log(p + eps) : log(1.0 - p + eps));
      int pred = (p > 0.5) ? 1 : 0;
      if (pred == y[i]) correct++;
    }
    loss /= (double)N;

    /* 勾配 grad[d] = (1/N) Σ_i err[i] * x_i[d]。特徴ごとに独立なので d で並列化 (競合なし)。 */
#pragma omp parallel for
    for (int d = 0; d < D; d++) {
      double g = 0.0;
      for (long i = 0; i < N; i++) g += err[i] * X[i * D + d];
      grad[d] = g / (double)N;
    }
    /* 重みの更新 */
    for (int d = 0; d < D; d++) w[d] -= lr * grad[d];

    if (ep % 50 == 0 || ep == E - 1)
      printf("epoch %3d: loss=%.4f, acc=%.2f%%\n", ep, loss, 100.0 * correct / N);
  }
  double elapsed = omp_get_wtime() - t0;

  printf("最終: N=%ld, D=%d, epochs=%d, loss=%.4f, acc=%.2f%%\n",
         N, D, E, loss, 100.0 * correct / N);
  printf("elapsed = %.3f sec\n", elapsed);
  free(w_true); free(X); free(y); free(w); free(grad); free(err);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore regression.cpp -o regression_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./regression_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ regression.f90
module logreg_mod
contains
  ! 状態を持たない乱数 (合成データ生成用): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  function sigmoid(z) result(s)
    real(8), intent(in) :: z
    real(8) :: s
    s = 1.0d0 / (1.0d0 + exp(-z))
  end function sigmoid
end module logreg_mod

! 勾配降下法でロジスティック回帰を学習する。
! 予測 p = sigmoid(w・x)。損失は二値クロスエントロピー。
! 勾配 grad(jd) = (1/N) Σ_i (sigmoid(w・x_i) - y_i) * x_i(jd)。
! w を 0 から始め, 各エポックで w(jd) -= lr * grad(jd) と更新する。
! 合成データは線形分離可能なので, 学習が進むと正解率が上がっていくのを観察できる。
! 並列化対象は「全サンプルにわたる予測・誤差の和」(行列積 + reduction)。
program regression
  use logreg_mod
  use omp_lib
  character(len=32) :: arg
  integer :: D, E, ep, jd, predc
  integer(8) :: N, i
  real(8) :: lr, loss, z, p, eps, score, xv, g, t0, elapsed
  integer(8) :: correct
  real(8), allocatable :: w_true(:), X(:), w(:), grad(:), err(:)
  integer, allocatable :: y(:)

  N = 200000_8; D = 20; E = 200; lr = 1.0d0
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) D
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) E
  end if
  if (command_argument_count() >= 4) then
     call get_command_argument(4, arg); read (arg, *) lr
  end if

  ! 真の重み w_true (= 学習で復元したい正解), 範囲 [-1,1)。添字 0 始まり。
  allocate(w_true(0:D-1), w(0:D-1), grad(0:D-1))
  allocate(X(0:N*D-1), y(0:N-1), err(0:N-1))
  do jd = 0, D - 1
     w_true(jd) = draw_rand01(int(jd,8), 7_8) * 2.0d0 - 1.0d0
  end do

  ! 特徴 x[i][jd] (中心化), ラベル y(i) = (w_true・x_i > 0)。線形分離可能。
  do i = 0, N - 1
     score = 0.0d0
     do jd = 0, D - 1
        xv = draw_rand01(i, int(jd,8)) - 0.5d0
        X(i*D + jd) = xv
        score = score + w_true(jd) * xv
     end do
     if (score > 0.0d0) then
        y(i) = 1
     else
        y(i) = 0
     end if
  end do

  do jd = 0, D - 1
     w(jd) = 0.0d0
  end do
  eps = 1.0d-12

  t0 = omp_get_wtime()
  do ep = 0, E - 1
     loss = 0.0d0
     correct = 0_8
     ! 各サンプルの予測 p = sigmoid(w・x_i), 誤差 err(i) = p - y_i,
     ! 損失・正解数を集計する。各サンプルは独立なので並列化できる。
     ! TODO: サンプルのループを !$omp parallel do reduction(+:loss,correct) で並列化せよ.
     do i = 0, N - 1
        z = 0.0d0
        do jd = 0, D - 1
           z = z + w(jd) * X(i*D + jd)
        end do
        p = sigmoid(z)
        err(i) = p - real(y(i), 8)
        if (y(i) == 1) then
           loss = loss - log(p + eps)
        else
           loss = loss - log(1.0d0 - p + eps)
        end if
        if (p > 0.5d0) then
           predc = 1
        else
           predc = 0
        end if
        if (predc == y(i)) correct = correct + 1
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
     loss = loss / real(N, 8)

     ! 勾配 grad(jd) = (1/N) Σ_i err(i) * x_i(jd)。特徴ごとに独立なので jd で並列化 (競合なし)。
     !$omp parallel do private(jd, i, g)
     do jd = 0, D - 1
        g = 0.0d0
        do i = 0, N - 1
           g = g + err(i) * X(i*D + jd)
        end do
        grad(jd) = g / real(N, 8)
     end do
     !$omp end parallel do
     ! 重みの更新
     do jd = 0, D - 1
        w(jd) = w(jd) - lr * grad(jd)
     end do

     if (mod(ep, 50) == 0 .or. ep == E - 1) then
        print "(a,i3,a,f0.4,a,f0.2,a)", &
             "epoch ", ep, ": loss=", loss, ", acc=", 100.0d0 * correct / N, "%"
     end if
  end do
  elapsed = omp_get_wtime() - t0

  print "(a,i0,a,i0,a,i0,a,f0.4,a,f0.2,a)", &
       "最終: N=", N, ", D=", D, ", epochs=", E, &
       ", loss=", loss, ", acc=", 100.0d0 * correct / N, "%"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program regression

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore regression.f90 -o regression_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./regression_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:regression.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:regression.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:regression.cpp}